# GO-GPT v2: inference

GO-GPT predicts Gene Ontology terms for a protein from its sequence and organism, in all three
aspects: Molecular Function (MF), Biological Process (BP) and Cellular Component (CC).

Install the package first (from the repository root): `pip install -e gogpt`.
A GPU is recommended but not required. The first run downloads GO-GPT (~0.75 GB) and the
ESM2-650M protein encoder (~2.6 GB) from the Hugging Face Hub.

In [1]:
import sys
from pathlib import Path

GOGPT_DIR = Path.cwd().parent  # this notebook lives in gogpt/notebooks/
try:
    import gogpt  # noqa: F401
except ImportError:  # not installed: use the source tree
    sys.path.insert(0, str(GOGPT_DIR / "src"))

from gogpt import GOGPTPredictor
from gogpt.utils.go_ontology import format_predictions, load_go_names

## Load the model

In [2]:
predictor = GOGPTPredictor.from_pretrained("wanglab/gogpt-v2")
print(f"device: {predictor.device}, precision: {predictor.precision}")

go_names = load_go_names(GOGPT_DIR / "data" / "go-basic.obo")

device: cuda, precision: bf16


## Example proteins

Three proteins from the held-out test split of `wanglab/bioreason-pro-v2`, so GO-GPT never saw
them in training. The organism is given as an NCBI taxonomy id.

In [3]:
EXAMPLES = [
    {
        "id": "Q9NPG2",
        "organism": 9606,  # Homo sapiens (Human)
        "sequence": "MERPEPELIRQSWRAVSRSPLEHGTVLFARLFALEPDLLPLFQYNCRQFSSPEDCLSSPEFLDHIRKVMLVIDAAVTNVEDLSSLEEYLASLGRKHRAVGVKLSSFSTVGESLLYMLEKCLGPAFTPATRAAWSQLYGAVVQAMSRGWDGE",
    },
    {
        "id": "P07821",
        "organism": 83333,  # Escherichia coli (strain K12)
        "sequence": "MQEYTNHSDTTFALRNISFRVPGRTLLHPLSLTFPAGKVTGLIGHNGSGKSTLLKMLGRHQPPSEGEILLDAQPLESWSSKAFARKVAYLPQQLPPAEGMTVRELVAIGRYPWHGALGRFGAADREKVEEAISLVGLKPLAHRLVDSLSGGERQRAWIAMLVAQDSRCLLLDEPTSALDIAHQVDVLSLVHRLSQERGLTVIAVLHDINMAARYCDYLVALRGGEMIAQGTPAEIMRGETLEMIYGIPMGILPHPAGAAPVSFVY",
    },
    {
        "id": "C5I9W9",
        "organism": 3702,  # Arabidopsis thaliana (Mouse-ear cress)
        "sequence": "METVSAVNQTLPISGGEPVKFTTYSAAVHKVLVMINAGILGLLQLVSQQSSVLETHKAAFLCFCVFILFYAVLRVREAMDVRLQPGLVPRLIGHGSHLFGGLAALVLVSVVSTAFSIVLFLLWFIWLSAVVYLETNKPSACPPQLPPV",
    },
]

## Predict

`predict_batch` returns one dict per protein with the GO terms of each aspect. Terms come in the
order GO-GPT generated them, which goes from general to specific; the list includes the ancestors
of the specific terms, as in the training labels. Below, the five most specific terms per aspect.

In [4]:
predictions = predictor.predict_batch(
    [e["sequence"] for e in EXAMPLES],
    organisms=[e["organism"] for e in EXAMPLES],
)

for example, pred in zip(EXAMPLES, predictions):
    print(f"\n{example['id']} ({predictor.organism_mapper.names.get(str(example['organism']))})")
    for aspect in ("MF", "BP", "CC"):
        terms = pred[aspect]
        print(f"  {aspect}: {len(terms)} terms")
        for go_id in terms[-5:]:
            print(f"    {go_id}  {go_names.get(go_id, 'Unknown')}")


Q9NPG2 (Homo sapiens (Human))
  MF: 16 terms
    GO:0019825  oxygen binding
    GO:0060589  nucleoside-triphosphatase regulator activity
    GO:0030695  GTPase regulator activity
    GO:0098809  nitrite reductase activity
    GO:0005092  GDP-dissociation inhibitor activity
  BP: 20 terms
    GO:0071453  cellular response to oxygen levels
    GO:0001666  response to hypoxia
    GO:0015671  oxygen transport
    GO:0036294  cellular response to decreased oxygen levels
    GO:0071456  cellular response to hypoxia
  CC: 16 terms
    GO:0070013  intracellular organelle lumen
    GO:0005634  nucleus
    GO:0005739  mitochondrion
    GO:0031981  nuclear lumen
    GO:0005654  nucleoplasm

P07821 (Escherichia coli (strain K12))
  MF: 4 terms
    GO:0003674  molecular function
    GO:0005215  transporter activity
    GO:0022857  transmembrane transporter activity
    GO:0015343  siderophore-iron transmembrane transporter activity
  BP: 39 terms
    GO:0000041  transition metal ion transport
    

## Organisms

GO-GPT has an embedding for each of the 200 organisms most frequent in its training data. Pass
the organism as a taxonomy id (`9606` or `"9606"`) or by name (`"Homo sapiens (Human)"` or
`"Homo sapiens"`). Any other organism, or none, uses a shared "unknown organism" embedding and
raises a warning.

In [5]:
names = predictor.organism_mapper.names
print(f"{len(names)} organisms, e.g.:", [names[k] for k in ("9606", "10090", "559292", "83333")])

human = EXAMPLES[0]["sequence"]
assert predictor.predict(human, organism=9606) == predictor.predict(human, organism="Homo sapiens")

200 organisms, e.g.: ['Homo sapiens (Human)', 'Mus musculus (Mouse)', "Saccharomyces cerevisiae (strain ATCC 204508 / S288c) (Baker's yeast)", 'Escherichia coli (strain K12)']


## Text for BioReason-Pro 2

`format_predictions` renders predictions as the GO-term block BioReason-Pro 2 reads in its
prompt: one line per aspect, each term followed by its name.

In [6]:
print(format_predictions(predictions[0], go_names))

Molecular Function (MF): GO:0003674 (molecular function), GO:0003824 (catalytic activity), GO:0005488 (binding), GO:0098772 (molecular function regulator activity), GO:0140104 (molecular carrier activity), GO:0005344 (oxygen carrier activity), GO:0005515 (protein binding), GO:0016491 (oxidoreductase activity), GO:0030234 (enzyme regulator activity), GO:0036094 (small molecule binding), GO:0016661 (oxidoreductase activity, acting on other nitrogenous compounds as donors), GO:0019825 (oxygen binding), GO:0060589 (nucleoside-triphosphatase regulator activity), GO:0030695 (GTPase regulator activity), GO:0098809 (nitrite reductase activity), GO:0005092 (GDP-dissociation inhibitor activity)
Biological Process (BP): GO:0008150 (biological process), GO:0009987 (cellular process), GO:0050896 (response to stimulus), GO:0051179 (localization), GO:0006950 (response to stress), GO:0009628 (response to abiotic stimulus), GO:0042221 (response to chemical), GO:0051234 (establishment of localization), 

## Decoding options

The default is greedy decoding, which is deterministic and is what the reported results use.
Beam search and sampling are also available:

In [7]:
beam = predictor.predict(human, organism=9606, decoding="beam", beam_size=5)
sampled = predictor.predict(human, organism=9606, decoding="sample", temperature=0.7, top_k=50)
for name, pred in (("greedy", predictions[0]), ("beam", beam), ("sampled", sampled)):
    print(f"{name:8s}", {aspect: len(terms) for aspect, terms in pred.items()})

greedy   {'MF': 16, 'BP': 20, 'CC': 16}
beam     {'MF': 16, 'BP': 20, 'CC': 15}
sampled  {'MF': 16, 'BP': 20, 'CC': 16}
